In [1]:
import sys
import json
from tqdm import tqdm

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.qa_pipeline import QAPipeline
from src.qa_pipeline.query_parser import QueryLLMParser
from src.qa_pipeline.knowledge_comparator import KnowledgeComparator
from src.qa_pipeline.knowledge_retriever import KnowledgeRetriever
from src.qa_pipeline.answer_generator import QALLMGenerator

from src.llm_agent import AgentConnector
from src.knowledge_graph_model import KnowledgeGraphModel
from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection

from src.embedding_functions import ChromaConnection, VectorDBConnectionConfig, EmbeddingsDatabaseConnectionConfig

In [2]:
agent = AgentConnector.open()
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://31.207.47.254:7687", user="neo4j", pwd="password", db_name="testdb"),
    embeddings_db=EmbeddingsDatabaseConnection()
)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with MEAN pooling.


In [3]:
QUERY = "Which device is better in battery life: iPhone11 Pro Max or Xiaomi 11?"

### QA-пайплан целиком

In [8]:
qa_pipeline = QAPipeline(kg_model, agent)
qa_pipeline.answer(QUERY)

'iPhone11 Pro Max'

### QA-пайплайн по частям

In [4]:
# stage 1
q_parser = QueryLLMParser(agent)
qparser_out = q_parser.extract_entities(QUERY)
print(qparser_out.entities)

['device', 'battery life', 'iPhone11 Pro Max', 'Xiaomi 11']


In [5]:
# stage 2
k_comparator = KnowledgeComparator(kg_model)
k_comparator.link_kgnodes_to_query(qparser_out)

for item in qparser_out.linked_nodes:
    print(item)

VectorDBInstance(id='4:d958299b-8cff-4454-876f-4f337d0518bd:54', document='vivo (kind: device)', embedding=[0.0544700101017952, -0.0022342100273817778, -0.04060351848602295, -0.06744229048490524, 0.06118621677160263, -0.016234172508120537, 0.008561200462281704, 0.04780689254403114, 0.01361271645873785, 0.006882958114147186, 0.051005467772483826, -0.026899440214037895, 0.0468338280916214, -0.038566600531339645, -0.031479574739933014, 0.01321228127926588, 0.028008535504341125, -0.008242599666118622, -0.03068714775145054, -0.04450353607535362, 0.06932621449232101, -0.014589780010282993, -0.051303643733263016, 0.03551040217280388, 0.05624421685934067, 0.03291082754731178, -0.049993377178907394, 0.003509256988763809, 0.0461731031537056, -0.04635797068476677, -0.056581318378448486, -0.029357975348830223, 0.04765455424785614, -0.03948742896318436, 0.10030793398618698, 0.056765489280223846, -0.04506104812026024, -0.05203470587730408, 0.04917019233107567, -0.03191229701042175, -0.00550386775285

In [6]:
# stage 3
k_retriever = KnowledgeRetriever(kg_model)
kretriever_out = k_retriever.retrieve(qparser_out)

for item in kretriever_out:
    print(item)

Triplet(start_node=Node(name='iphone11_pro_max', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:16', prop={'kind': 'device', 'name': 'iphone11_pro_max'}, stringified='iphone11_pro_max (kind: device)'), relation=Relation(name='opinion', type=<RelationType.simple: 'simple'>, id='5:d958299b-8cff-4454-876f-4f337d0518bd:3881', prop={'raw_time': '24685', 'sentiment': 'neu', 'person': 'Dennis', 'name': 'opinion', 'time': '26.10.2020', 'opinion': 'decent_battery_life'}), end_node=Node(name='battery_life', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:17', prop={'kind': 'feature', 'name': 'battery_life'}, stringified='battery_life (kind: feature)'), id='49ca6f928757b92624359a2eee6f9fa7', stringified=None)
Triplet(start_node=Node(name='iphone11_pro_max', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:16', prop={'kind': 'device', 'name': 'iphone11_pro_max'}, stringified='iphone11_pro_max (kind: device)'), r

In [7]:
# stage 4
qa_generator = QALLMGenerator(agent)
contexts = qa_generator.formate_context(kretriever_out)
qagenerator_out = qa_generator.generate(qparser_out.query, contexts)

print(contexts)
print()
print(qagenerator_out)

- 26.10.2020: iphone11_pro_max (kind: device) opinion (sentiment: neu; person: Dennis; opinion: decent_battery_life) battery_life (kind: feature)
- 15.12.2020: iphone11_pro_max (kind: device) opinion (sentiment: pos; person: Rita; opinion: last_a_day) battery_life (kind: feature)
- 15.9.2018: iphone11_pro_max (kind: device) opinion (sentiment: pos; person: Dennis; opinion: first_ladder) battery_life (kind: feature)
- 10.3.2020: xiaomi_11 (kind: device) opinion (sentiment: pos; person: Andrew; opinion: completely_no_color_diviation) screen (kind: feature)
- 27.5.2020: xiaomi_11 (kind: device) opinion (sentiment: pos; person: Rodrigo; opinion: xiaomi_11s_beast) battery_life (kind: feature)
- 10.12.2020: xiaomi_11 (kind: device) opinion (sentiment: neu; person: Rodrigo; opinion: decent_not_great) battery_life (kind: feature)
- 10.12.2020: xiaomi_11 (kind: device) opinion (sentiment: neg; person: Brianna; opinion: drains_so_fast) battery_life (kind: feature)
- 21.8.2020: xiaomi_11 (kind: d